In [1]:
import pandas as pd 
import numpy as np

In [2]:
X_test = pd.read_csv('../Data/Sampled_Data/X_test.csv')

y_test = pd.read_csv('../Data/Sampled_Data/y_test.csv').squeeze()  

y_test_bin = (y_test != 0).astype(int)
print(pd.Series(y_test_bin).value_counts())

0
0    454265
1    111311
Name: count, dtype: int64


In [3]:
models = {
    "RF_Baseline": "../Data/Sampled_Data/rf.pkl",
    "RF_SMOTE": "../Data/Sampled_Data/rf_smote.pkl",
    "RF_SMOTE_KMeans": "../Data/Sampled_Data/rf_smote_kmeans.pkl",
    "RF_SMOTE_Borderline": "../Data/Sampled_Data/rf_smote_bl.pkl",
    "XGB_SMOTE": "../Data/Sampled_Data/XG_smote.pkl",
    "XGB_SMOTE_KMeans": "../Data/Sampled_Data/XG_smote_kmeans.pkl",
    "XGB_SMOTE_Borderline": "../Data/Sampled_Data/XG_smote_BL.pkl"
}

In [4]:
import joblib
import gc
from sklearn.metrics import *

results = []

for name, path in models.items():
    
    model = joblib.load(path)
    
    y_prob = model.predict_proba(X_test.values)[:, 1]
    y_pred = (y_prob > 0.28).astype(int)
    
    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test_bin, y_pred),
        "Precision": precision_score(y_test_bin, y_pred),
        "Recall": recall_score(y_test_bin, y_pred),
        "F1": f1_score(y_test_bin, y_pred),
        "ROC_AUC": roc_auc_score(y_test_bin, y_prob),
        "PR_AUC": average_precision_score(y_test_bin, y_prob)
    }
    
    results.append(metrics)
    
    del model
    gc.collect()

In [17]:
baseline_rf_shap = pd.read_csv("../Data/Sampled_Data/rf_shap.csv")
print(baseline_rf_shap.shape)
print(X_test.shape)
smote_rf_shap = pd.read_csv("../Data/Sampled_Data/rf_smote_full_shap.csv")
print(smote_rf_shap.shape)

(1500, 78)
(565576, 78)
(1500, 78)


In [9]:
smote_importance = pd.read_csv("../Data/Sampled_Data/rf_smote_shap_importance.csv")
print(smote_importance.shape)

baseline_importance = pd.read_csv("../Data/Sampled_Data/rf_baseline_shap_importance.csv")
print(baseline_importance.shape)



(78, 2)
(78, 2)


In [10]:
from scipy.stats import spearmanr

corr, pval = spearmanr(
    baseline_importance["importance"],
    smote_importance["importance"]
)

print("Spearman Correlation:", corr)
print("p-value:", pval)

Spearman Correlation: 1.0
p-value: 0.0


In [ ]:
print("Overall Magnitude:", np.mean(np.abs(baseline_rf_shap)))

Overall Magnitude: 0.007110072405893653


In [22]:
from scipy.stats import ttest_ind


t_stat, p_val = ttest_ind(
    np.abs(baseline_rf_shap.values).flatten(),
    np.abs(smote_rf_shap.values).flatten(),
    equal_var=False
)

print("Global p-value:", p_val)

Global p-value: 1.0


In [23]:
baseline_mean = np.mean(np.abs(baseline_rf_shap), axis=0)
smote_mean = np.mean(np.abs(smote_rf_shap), axis=0)

t_stat, p_val = ttest_ind(
    baseline_mean,
    smote_mean,
    equal_var=False
)

print("Importance-level p-value:", p_val)

Importance-level p-value: 1.0


In [24]:
from scipy.stats import spearmanr

corr, _ = spearmanr(baseline_mean, smote_mean)
print("Rank correlation:", corr)

print("Baseline magnitude:", np.mean(np.abs(baseline_rf_shap)))
print("SMOTE magnitude:", np.mean(np.abs(smote_rf_shap)))

print("Baseline variance:", baseline_rf_shap.var())
print("SMOTE variance:", smote_rf_shap.var())

Rank correlation: 1.0
Baseline magnitude: 0.007110072405893653
SMOTE magnitude: 0.007110072405893653
Baseline variance: 0     0.002937
1     0.000023
2     0.000018
3     0.000202
4     0.000073
        ...   
73    0.000004
74    0.000053
75    0.000003
76    0.000010
77    0.000028
Length: 78, dtype: float64
SMOTE variance: 0     0.002937
1     0.000023
2     0.000018
3     0.000202
4     0.000073
        ...   
73    0.000004
74    0.000053
75    0.000003
76    0.000010
77    0.000028
Length: 78, dtype: float64


In [33]:
xgb_smote_importance = pd.read_csv("../Data/Sampled_Data/XG_smote_importance.csv")
print(xgb_smote_importance.shape)
xgb_smote_kmeans_importance = pd.read_csv("../Data/Sampled_Data/XG_smote_kmeans_importance.csv")
print(xgb_smote_kmeans_importance.shape)
xgb_smote_bl_importance = pd.read_csv("../Data/Sampled_Data/XG_smote_bl_importance.csv")
print(xgb_smote_bl_importance.shape)

(78, 2)
(78, 2)
(78, 2)


In [34]:
from scipy.stats import spearmanr

corr, pval = spearmanr(
    xgb_smote_importance["importance"],
    xgb_smote_kmeans_importance["importance"]
)

print("Spearman Correlation:", corr)
print("p-value:", pval)

Spearman Correlation: 0.999581443999802
p-value: 1.0534242630992794e-118


In [36]:
baseline_arr = baseline_rf_shap.to_numpy()
smote_arr = smote_rf_shap.to_numpy()

In [37]:
mean_shift = np.mean(np.abs(baseline_arr - smote_arr))
print("Mean local SHAP shift:", mean_shift)

Mean local SHAP shift: 0.0


In [38]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = [
    cosine_similarity(
        baseline_arr[i].reshape(1,-1),
        smote_arr[i].reshape(1,-1)
    )[0][0]
    for i in range(len(baseline_arr))
]

print("Mean cosine similarity:", np.mean(similarities))

Mean cosine similarity: 1.0
